# LHSD vs. Calibrated Geometry — Controlled Benchmark

**Question:** Does the existing noise-aware, conformal upper-bound scale selector improve geometry recovery over an LHSD spectral baseline using **the same frozen score network**?

**No adversarial attacks. No new geometry estimator.** The only proposed method change is the already-developed calibration/selection wrapper.

**What this notebook rigorously implements**
- Official LHSD **Soft_DoF spectral head** (Osada, ICML 2026) via an **exact 8-dimensional trace** of its published spectral function, evaluated on the same normalized Hessian/score-Jacobian `Q` used by our synthetic notebook. This removes SLQ approximation noise at D=8. It does **not** reproduce the entire published LHSD experimental pipeline or an author-endorsed noise-scale selection algorithm.
- Optional sanity check using the **exact official repository Soft_DoF function**, checked out at a pinned commit. This is separate from the fast exact-trace main benchmark.
- Our existing `Noise_Aware_Tangent_Selector.py` unchanged: point-level train/conformal/test split, Ridge/RF comparison, simultaneous-scale upper bound, select/abstain.
- Common tangent extraction for *every* policy: smallest eigenvalue directions from the **same learned Q**, using true rank **for evaluation only**. This yields an oracle-rank tangent diagnostic, **not** a real-data tangent guarantee.

**Required precomputed artifacts** from `Ground_Truth_Tangent_Recovery_Cross_Manifold_Extended.ipynb`:
`circle_per_point_results.csv`, `sphere_per_point_results.csv` etc., and corresponding `*_eps*.pt` checkpoints under `MyDrive/synthetic_tangent_ground_truth`.

Official links: [LHSD paper](https://proceedings.mlr.press/v306/osada26a.html) · [Official GitHub](https://github.com/geosada/LHSD) · [Spectral source](https://github.com/geosada/LHSD/blob/main/lid/diffusions/local_hessian_spectrum.py).

**Interpretation limit:** Conformal true-error coverage holds marginally for exchangeable points from a fixed synthetic family with independent train/cal/test samples. It is not a guarantee on MNIST, on unseen manifolds, or conditional on acceptance.

## 1. Configuration — keep published LHSD head and selection protocol fixed

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, torch, json, os, random, hashlib, math, time
import matplotlib.pyplot as plt
from IPython.display import display

SEED=73
SIGMAS=[0.030, 0.128, 0.342]
D=8
N_FIT=40
N_TEST=100
# Start with circle. Add "sphere", "ellipse", "torus" when ready.
FAMILIES_TO_RUN=["circle", "sphere"]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
# Exact trace for D=8; the official SLQ function is verified in an optional cell.
LHSD_SOFT_C=0.5
LHSD_SOFT_P=12
PRESET_FIXED_SIGMA=0.128  # chosen *before* consulting test labels
ALPHA=0.10
TRUE_ERROR_TAU=0.15  # preregistered for synthetic oracle-rank tangent evaluation
VERIFY_OFFICIAL_SLQ=False  # Optional 2-point source-code cross-check; uses internet

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR=Path('/content/drive/MyDrive/synthetic_tangent_ground_truth')
    RESULT_DIR=Path('/content/drive/MyDrive/LHSD_vs_Calibrated_Geometry_Benchmark')
except ImportError:
    DATA_DIR=Path('/mnt/data/synthetic_tangent_ground_truth')
    RESULT_DIR=Path('/mnt/data/LHSD_vs_Calibrated_Geometry_Benchmark')
RESULT_DIR.mkdir(parents=True,exist_ok=True)
assert DATA_DIR.is_dir(), f'Missing {DATA_DIR}; run the extended synthetic recovery notebook first'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.set_num_threads(min(4,os.cpu_count() or 2))
print('DEVICE:',DEVICE,'DATA_DIR:',DATA_DIR)

## 2. Source-verified LHSD Soft_DoF spectral function
The [official source](https://github.com/geosada/LHSD/blob/main/lid/diffusions/local_hessian_spectrum.py) defines `Soft_DoF` with a Hill-type head `1 / (1 + (max(λ,0)/κ)**p)` and `κ=c/σ²`. With our normalized `Q=-σ² Hessian(log p)`, `σ²λ/ c = λ_Q/c`. **For these 8D experiments**, compute the exact spectral trace instead of SLQ; this holds the spectral estimator fixed and removes Monte-Carlo variance. Never refer to this shortcut as an end-to-end official LHSD reproduction.

In [ ]:
def lhsd_soft_dof_from_Q(Q,soft_c=LHSD_SOFT_C,p=LHSD_SOFT_P):
    """Exact eigen-trace of *official* LHSD Soft_DoF spectral head, D=8 only.
    This is a mathematical equivalent to the full-probe expectation of Soft_DoF,
    NOT its published stochastic Lanczos implementation or full experiment.
    """
    lam=np.linalg.eigvalsh(0.5*(Q+np.swapaxes(Q,-1,-2)))
    t=np.maximum(lam,0.0)/soft_c
    return np.sum(1.0/(1.0+t**p),axis=-1)

# Exact numerical sanity check for a 2-tangent/6-normal spectrum.
assert np.isclose(lhsd_soft_dof_from_Q(np.diag([0,0,1,1,1,1,1,1])),2+6/(1+2**12))
print('Exact LHSD Soft_DoF spectral-head test: PASS')

## 3. Restore the original synthetic model architecture and manifold parametrizations
The existing checkpoints are **not** retrained. The embeddings, noise scales, architecture, and seeds below mirror the extended synthetic-recovery notebook. Feature/test identity is audited before any inference claims.

In [ ]:
from torch import nn
class EpsNetwork(nn.Module):
    def __init__(self,D=8,width=128):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(D+1,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,D))
    def forward(self,y,sigma):
        if not isinstance(sigma,torch.Tensor):
            sigma=torch.full((len(y),1),float(sigma),device=y.device,dtype=y.dtype)
        if sigma.ndim==1: sigma=sigma[:,None]
        return self.net(torch.cat([y,torch.log(sigma)],dim=1))

_rng=np.random.default_rng(SEED)
B,_=np.linalg.qr(_rng.normal(size=(D,2)))
_vec=np.random.default_rng(SEED+101).normal(size=D)
_vec=_vec-B@(B.T@_vec)
_vec/=np.linalg.norm(_vec)
B3=np.column_stack([B,_vec])
R=1.0; A_ELL,B_ELL=1.30,0.70;R_SPHERE=1.0;R_MAJOR,R_MINOR=1.20,0.45
FAMILIES=('ellipse','sphere','torus')
OFFSET=np.array([-0.14,-0.08,0.08,0.14])

In [ ]:
def sample_family_params(family, n, rng):
    if family == 'ellipse':
        return rng.uniform(0, 2*np.pi, size=(n,1))
    if family == 'sphere':
        return np.column_stack([rng.uniform(0,2*np.pi,size=n), rng.uniform(-1,1,size=n)])
    if family == 'torus':
        return rng.uniform(0,2*np.pi,size=(n,2))
    raise ValueError(family)

def family_points_and_tangent(family, params):
    # Return x:(n,D), true orthonormal tangent rows:(n,d,D).
    p = np.asarray(params,dtype=float)
    if family=='ellipse':
        th=p[:,0]; c=np.cos(th); s=np.sin(th)
        local=np.column_stack([A_ELL*c, B_ELL*s])
        x=local@B.T
        v=(-A_ELL*s)[:,None]*B[:,0]+(B_ELL*c)[:,None]*B[:,1]
        return x, (v/np.linalg.norm(v,axis=1,keepdims=True))[:,None,:]
    if family=='sphere':
        th,z=p[:,0],p[:,1]
        a=np.sqrt(np.clip(1-z*z,1e-12,None))
        local=R_SPHERE*np.column_stack([a*np.cos(th),a*np.sin(th),z])
        t1=np.column_stack([-np.sin(th),np.cos(th),np.zeros_like(th)])
        t2=np.column_stack([-z*np.cos(th),-z*np.sin(th),a])
        x=local@B3.T
        T=np.stack([t1@B3.T,t2@B3.T],axis=1)
        return x,T
    if family=='torus':
        u,v=p[:,0],p[:,1]
        cv,sv,cu,su=np.cos(v),np.sin(v),np.cos(u),np.sin(u)
        radial=R_MAJOR+R_MINOR*cv
        local=np.column_stack([radial*cu,radial*su,R_MINOR*sv])
        # Coordinate derivative directions normalized to length 1.
        t1=np.column_stack([-su,cu,np.zeros_like(u)])
        t2=np.column_stack([-sv*cu,-sv*su,cv])
        return local@B3.T, np.stack([t1@B3.T,t2@B3.T],axis=1)
    raise ValueError(family)

def family_neighbor_chords(family,params,x,T):
    # Keep the original four offset magnitudes. For 2D, cover both tangent axes.
    n,d,_=T.shape
    neighbors=[]
    for j in range(d):
        for delta in OFFSET:
            if family=='ellipse':
                th=params[:,0]
                speed=np.sqrt((A_ELL*np.sin(th))**2+(B_ELL*np.cos(th))**2)
                shifted=params.copy(); shifted[:,0]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            elif family=='sphere':
                # Exact sphere great-circle step of arc length |delta|.
                x2=np.cos(delta/R_SPHERE)*x+(R_SPHERE*np.sin(delta/R_SPHERE))*T[:,j,:]
            elif family=='torus':
                u,v=params[:,0],params[:,1]
                speed=(R_MAJOR+R_MINOR*np.cos(v)) if j==0 else np.full(n,R_MINOR)
                shifted=params.copy(); shifted[:,j]+=delta/speed
                x2,_=family_points_and_tangent(family,shifted)
            vec=x2-x
            neighbors.append(vec/(np.linalg.norm(vec,axis=1,keepdims=True)+1e-12))
    return np.stack(neighbors,axis=1)

def generalized_diagnostics(Q,T,x,params,family,rng):
    # Same original proxies; projector error normalized by sqrt(true tangent dim).
    Q=0.5*(Q+np.swapaxes(Q,-1,-2))
    eig,V=np.linalg.eigh(Q)
    n,d,Dlocal=T.shape
    P_hat=V[:,:,:d] @ np.swapaxes(V[:,:,:d],-1,-2)
    P_true=np.swapaxes(T,-1,-2)@T
    overlap=np.einsum('nij,nij->n',P_hat,P_true)
    err=np.sqrt(np.maximum(0.0,2.0*(d-overlap)/d))
    defect=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
    nb=family_neighbor_chords(family,params,x,T)
    rand=rng.normal(size=(n,16,Dlocal))
    rand/=np.linalg.norm(rand,axis=2,keepdims=True)+1e-12
    nb_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,nb)**2,axis=-1),axis=1)
    rand_energy=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,rand)**2,axis=-1),axis=1)
    proxy=np.minimum(1.0,nb_energy/(rand_energy+1e-12))
    soft_dim=np.sum(1-np.clip(eig,0,1),axis=1)
    return err,defect,proxy,soft_dim

In [ ]:
def circle_point_data(split):
    theta=np.random.default_rng(SEED+(1 if split=='fit' else 2)).uniform(0,2*np.pi,
        N_FIT if split=='fit' else N_TEST)
    coords=np.cos(theta)[:,None]*B[:,0][None,:]+np.sin(theta)[:,None]*B[:,1][None,:]
    tangent=(-np.sin(theta)[:,None]*B[:,0][None,:]+np.cos(theta)[:,None]*B[:,1][None,:])[:,None,:]
    indices=np.arange(N_FIT if split=='fit' else N_TEST)+(0 if split=='fit' else N_FIT)
    return indices,coords,tangent,theta[:,None]

def family_data(fam,split):
    if fam=='circle':return circle_point_data(split)
    assert fam in FAMILIES
    fi=FAMILIES.index(fam)
    seed=SEED+300+10*fi+(0 if split=='fit' else 1)
    n=N_FIT if split=='fit' else N_TEST
    p=sample_family_params(fam,n,np.random.default_rng(seed))
    x,T=family_points_and_tangent(fam,p)
    indices=np.arange(n)+(0 if split=='fit' else N_FIT)
    return indices,x,T,p

def checkpoint_for(fam):
    if fam=='circle':return DATA_DIR/'synthetic_eps_network.pt'
    return DATA_DIR/f'{fam}_eps_R8_seed{SEED}_steps2500_v1.pt'

def get_model(fam):
    p=checkpoint_for(fam)
    assert p.exists(), f'Checkpoint missing: {p}. Run extended synthetic notebook first.'
    net=EpsNetwork(D).to(DEVICE)
    state=torch.load(p,map_location=DEVICE,weights_only=True)
    net.load_state_dict(state); net.eval()
    for v in net.parameters():v.requires_grad_(False)
    return net

@torch.no_grad()
def q_batches(model,x,sigma,h=0.001):
    X=np.asarray(x,dtype=np.float32)
    cols=[]
    for j in range(D):
        plus=X.copy();minus=X.copy()
        plus[:,j]+=h;minus[:,j]-=h
        y1=model(torch.from_numpy(plus).to(DEVICE),float(sigma)).cpu().numpy()
        y2=model(torch.from_numpy(minus).to(DEVICE),float(sigma)).cpu().numpy()
        cols.append((y1-y2)/(2*h))
    Q=float(sigma)*np.stack(cols,axis=-1)
    return 0.5*(Q+np.swapaxes(Q,-1,-2))

## 4. Generate the paired benchmark observations
This computes the **published LHSD Soft_DoF head** and shared tangent diagnostics from each frozen model on exactly the original fit/test points. Synthetic tangent ground truth is used only to create error labels, **never passed to the selector on test**. Do not substitute the previous `soft_dimension` column: it used a different spectral head.

In [ ]:
bench_files={}
for fam in FAMILIES_TO_RUN:
    input_file=DATA_DIR/f'{fam}_per_point_results.csv'
    assert input_file.exists(),f'Missing {input_file}'
    old=pd.read_csv(input_file)
    old=old[old.model=='learned_score'].copy()
    assert {'model','split','index','sigma','tangent_error','neighbor_proxy','projector_defect','soft_dimension'}<=set(old)
    net=get_model(fam)
    rows=[]
    for split in ['fit','test']:
        ids,X,T,params=family_data(fam,split)
        assert len(ids)==(N_FIT if split=='fit' else N_TEST)
        for sigma in SIGMAS:
            Q=q_batches(net,X,sigma)
            evals,V=np.linalg.eigh(Q)
            d_true=T.shape[1]
            Phat=V[:,:,:d_true]@np.swapaxes(V[:,:,:d_true],-1,-2)
            Ptrue=np.swapaxes(T,-1,-2)@T
            true_err=np.linalg.norm(Phat-Ptrue,axis=(1,2))/np.sqrt(d_true)
            lhsd_dim=lhsd_soft_dof_from_Q(Q)
            defect=np.linalg.norm(Q@Q-Q,axis=(1,2))/(np.linalg.norm(Q,axis=(1,2))+1e-12)
            for i,idx in enumerate(ids):
                old_row=old[(old.split==split)&(old['index']==idx)&np.isclose(old.sigma,sigma)]
                if len(old_row)!=1: raise AssertionError(f'Point mismatch {fam} {split} {idx} {sigma} rows {len(old_row)}')
                rr=old_row.iloc[0]
                # Recompute error from the checkpoint to catch any incorrect model/point alignment.
                if abs(float(rr.tangent_error)-float(true_err[i]))>0.02:
                    raise AssertionError(f'Stale checkpoint/point geometry: {fam}, index={idx}, sigma={sigma}; old={rr.tangent_error:.4f}, new={true_err[i]:.4f}')
                rows.append(dict(model='learned_score',split=split,index=int(idx),sigma=float(sigma),
                    tangent_error=float(true_err[i]),neighbor_proxy=float(rr.neighbor_proxy),
                    projector_defect=float(defect[i]),soft_dimension=float(lhsd_dim[i]),
                    dim_true=int(d_true),dim_error=float(abs(lhsd_dim[i]-d_true)),
                    old_soft_dimension=float(rr.soft_dimension),
                    spectral_head='official_LHSD_Soft_DoF_exact_trace',
                    checkpoint=str(checkpoint_for(fam))))
    df=pd.DataFrame(rows)
    assert df.groupby(['split','index']).sigma.nunique().eq(len(SIGMAS)).all()
    assert not df[['tangent_error','neighbor_proxy','projector_defect','soft_dimension']].isna().any().any()
    file=RESULT_DIR/f'{fam}_LHSD_shared_operator.csv'
    df.to_csv(file,index=False)
    bench_files[fam]=file
    print(fam,'ready:',len(df),'rows; means:',df[df.split=='test'].groupby('sigma')[['soft_dimension','tangent_error']].mean().round(4).to_dict())
    del net
    if DEVICE=='cuda':torch.cuda.empty_cache()

## 5. Reuse the original noise-aware selector code **unchanged**
This source is copied directly from the earlier `Noise_Aware_Tangent_Selector_Colab(1).ipynb`, keeping its 20/20 split, group-CV model selection, split-conformal maximum-over-scales score, candidate selection, and abstention. No new estimator is introduced.

In [ ]:
%%writefile /content/Noise_Aware_Tangent_Selector.py
"""Geometry scale selection with held-out true tangent errors.

Run:
python Noise_Aware_Tangent_Selector.py --csv all_per_point_results.csv --out results

No score-model training or inference. Uses saved synthetic observations only.
Statistical coverage is marginal for an exchangeable point from the same synthetic
setting (and with all three sigmas observed), not distribution-free on MNIST.
"""
import argparse, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

FEATS = ['sigma','neighbor_proxy','projector_defect','soft_dimension']
LABEL = 'tangent_error'
REQUIRED = {'model','split','index','sigma',LABEL,'neighbor_proxy','projector_defect','soft_dimension'}


def load_data(path):
    df = pd.read_csv(path)
    missing = REQUIRED-set(df.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    df = df[df['model']=='learned_score'].copy()
    if df.empty: raise ValueError('No rows with model=learned_score')
    if df[list(REQUIRED - {'model','split'})].isna().any().any():
        raise ValueError('Found missing numbers in required columns')
    if df.duplicated(['split','index','sigma']).any():
        raise ValueError('Duplicate point/sigma rows')
    for sp in ['fit','test']:
        chunk=df[df.split==sp]
        if chunk.empty: raise ValueError(f'Missing {sp} split')
        counts=chunk.groupby('index')['sigma'].nunique()
        if counts.nunique()!=1 or counts.iloc[0]!=df.sigma.nunique():
            raise ValueError(f'{sp} must contain the same complete sigma grid per point')
    return df


def create_splits(df,seed=73):
    fit_ids = sorted(df.loc[df.split=='fit','index'].unique())
    # deterministic image-wise split, not row-wise split
    rng = np.random.default_rng(seed+159)
    ordered = rng.permutation(fit_ids)
    ntrain = max(10, int(np.floor(0.5*len(ordered))))
    if len(ordered)-ntrain<9:
        raise ValueError('At least 19 distinct fitting points needed to allow train+conformal splits')
    train_ids=set(ordered[:ntrain]); cal_ids=set(ordered[ntrain:])
    train=df[(df.split=='fit')&df['index'].isin(train_ids)].copy()
    calib=df[(df.split=='fit')&df['index'].isin(cal_ids)].copy()
    test=df[df.split=='test'].copy()
    return train,calib,test


def choose_predictor(train,seed=73):
    candidates = {
        'ridge': make_pipeline(StandardScaler(),Ridge(alpha=3.0)),
        'random_forest': RandomForestRegressor(n_estimators=120,max_depth=5,
                   min_samples_leaf=3,random_state=seed,n_jobs=-1),
    }
    folds=min(5,train['index'].nunique())
    cv=GroupKFold(n_splits=folds)
    res={}
    for name,base in candidates.items():
        losses=[]
        for tr,va in cv.split(train[FEATS],train[LABEL],groups=train['index']):
            m=clone(base);m.fit(train.iloc[tr][FEATS],train.iloc[tr][LABEL]);
            pred=m.predict(train.iloc[va][FEATS]);
            losses.append(np.mean(np.abs(train.iloc[va][LABEL].to_numpy()-pred)))
        res[name]=float(np.mean(losses))
    selected=min(res,key=res.get)
    model=clone(candidates[selected]);model.fit(train[FEATS],train[LABEL])
    return model,selected,res


def add_pred(df,model):
    v=df.copy()
    v['predicted_error']=np.clip(model.predict(v[FEATS]),0,math.sqrt(2))
    return v


def calibrate(train, calib, alpha=.10):
    # sigma-dependent scale comes only from TRAINING residuals.
    per_sigma = train.assign(residual=lambda d:d[LABEL]-d['predicted_error']).groupby('sigma')\
        ['residual'].agg(lambda a:np.sqrt(np.mean(np.square(a))))
    floors={float(s):max(.01,float(v)) for s,v in per_sigma.items()}
    calib=calib.copy()
    calib['uncertainty_scale']=calib['sigma'].map(floors)
    calib['scaled_abs_residual']=np.abs(calib[LABEL]-calib['predicted_error'])/calib['uncertainty_scale']
    # A maximum over scales protects selection of ANY sigma after calibration.
    scores=calib.groupby('index')['scaled_abs_residual'].max().sort_values().to_numpy()
    n=len(scores)
    k=math.ceil((n+1)*(1-alpha))
    q=math.inf if k>n else float(scores[k-1])
    return floors,q,n,calib


def evaluate_selector(test,floors,q,tau):
    test=test.copy()
    test['uncertainty_scale']=test['sigma'].map(floors)
    test['predicted_upper_bound'] = test['predicted_error']+q*test['uncertainty_scale']
    test['simultaneous_bound_covered']=(test[LABEL]<=test['predicted_upper_bound'])
    methods={
        'raw_neighbor_proxy':'neighbor_proxy',
        'raw_projector_defect':'projector_defect',
        'noise_aware_predicted_error':'predicted_error',
        'calibrated_upper_bound':'predicted_upper_bound',
    }
    cols=['index','sigma',LABEL,'predicted_error','predicted_upper_bound',
          'neighbor_proxy','projector_defect','soft_dimension']
    pieces=[]
    for name,key in methods.items():
        z=test.loc[test.groupby('index')[key].idxmin(),cols].copy()
        z['method']=name
        z['accepted']=(z.predicted_upper_bound<=tau) if name=='calibrated_upper_bound' else True
        pieces.append(z)
    oracle=test.loc[test.groupby('index')[LABEL].idxmin(),cols].copy()
    oracle['method']='oracle_true_error_UNAVAILABLE_ON_MNIST';oracle['accepted']=True
    pieces.append(oracle)
    selected=pd.concat(pieces,ignore_index=True)
    summary=[]
    for name,z in selected.groupby('method'):
        acc=z[z.accepted]
        summary.append({
           'method':name,'coverage':float(len(acc)/len(z)),
           'accepted_count':int(len(acc)),
           'mean_true_error_accepted':float(acc[LABEL].mean()) if len(acc) else np.nan,
           'mean_true_error_all_selected':float(z[LABEL].mean()),
           'fraction_accepted_above_tau':float(np.mean(acc[LABEL]>tau)) if len(acc) else np.nan,
           'sigma_counts_accepted':json.dumps({str(k):int(v) for k,v in acc.sigma.value_counts().sort_index().items()}),
        })
    coverage_by_point=test.groupby('index')['simultaneous_bound_covered'].all()
    return selected,pd.DataFrame(summary).sort_values('mean_true_error_all_selected'),float(coverage_by_point.mean())


def pair_bootstrap(selected,alpha=.05,nboot=4000,seed=73):
    grid=selected.pivot(index='index',columns='method',values=LABEL)
    d=grid['raw_neighbor_proxy']-grid['calibrated_upper_bound']
    vals=d.to_numpy()
    rng=np.random.default_rng(seed+716)
    draws=rng.integers(0,len(vals),size=(nboot,len(vals)))
    boot=vals[draws].mean(axis=1)
    return float(vals.mean()), np.quantile(boot,[alpha/2,1-alpha/2]).tolist()


def run(csv,out,alpha=.10,tau=.15,seed=73):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    df=load_data(csv)
    tr,ca,te=create_splits(df,seed)
    model,name,cv=choose_predictor(tr,seed)
    tr=add_pred(tr,model)
    ca=add_pred(ca,model)
    te=add_pred(te,model)
    scales,q,n,ca=calibrate(tr,ca,alpha)
    selected,summary,observed= evaluate_selector(te,scales,q,tau)
    diff,ci=pair_bootstrap(selected,seed=seed)
    tr.to_csv(out/'train_points.csv',index=False)
    ca.to_csv(out/'calibration_points.csv',index=False)
    te.to_csv(out/'test_points_with_predictions.csv',index=False)
    selected.to_csv(out/'test_selected_by_method.csv',index=False)
    summary.to_csv(out/'method_comparison.csv',index=False)
    report={'source_csv':str(csv),'manifold':'circle R8, learned_score',
      'n_train_points':int(tr['index'].nunique()),'n_calib_points':int(ca['index'].nunique()),
      'n_test_points':int(te['index'].nunique()),'predictor':name,
      'train_only_cross_validation_MAE':cv,'sigmas':sorted(df.sigma.unique().tolist()),
      'alpha':alpha,'requested_error_tolerance':tau,'conformal_q':q,
      'sigma_residual_scales':{str(k):v for k,v in scales.items()},
      'observed_test_simultaneous_coverage':observed,
      'mean_paired_true_error_improvement_vs_raw_proxy':diff,
      'paired_bootstrap_CI95':ci,
      'limitation':'Coverage applies marginally to same-population points, NOT conditional on acceptance, NOT MNIST. The model uses true synthetic tangent labels in fitting and calibration.'}
    (out/'report.json').write_text(json.dumps(report,indent=2))
    print('Selection results on untouched test points:')
    print(summary.to_string(index=False))
    print('\n',json.dumps(report,indent=2))
    fig,axes=plt.subplots(1,2,figsize=(12,4.2))
    plot=summary[~summary['method'].str.contains('oracle')].sort_values('mean_true_error_all_selected')
    axes[0].barh(plot.method,plot.mean_true_error_all_selected)
    axes[0].set(xlabel='Mean TRUE tangent-projector error (all selected)',title='Held-out selection quality')
    axes[0].invert_yaxis()
    st=te.groupby('sigma').agg(true_error=(LABEL,'mean'),predicted_error=('predicted_error','mean'))
    axes[1].plot(st.index,st.true_error,marker='o',label='Actual error')
    axes[1].plot(st.index,st.predicted_error,marker='o',label='Predicted error')
    axes[1].set(xlabel='Noise sigma',ylabel='Mean tangent error',title='Scale-aware prediction (test)')
    axes[1].legend();axes[1].grid(alpha=.2)
    fig.tight_layout();fig.savefig(out/'selector_validation.png',dpi=170,bbox_inches='tight');plt.close(fig)
    print('Saved all reports to',out)
    return summary,report

if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--csv',required=True)
    p.add_argument('--out',default='noise_aware_selector_results')
    p.add_argument('--alpha',type=float,default=.1)
    p.add_argument('--tau',type=float,default=.15)
    p.add_argument('--seed',type=int,default=73)
    a=p.parse_args();run(a.csv,a.out,a.alpha,a.tau,a.seed)


In [ ]:
import sys
sys.path.insert(0,'/content' if Path('/content/Noise_Aware_Tangent_Selector.py').exists() else str(Path.cwd()))
# In ordinary Colab, the preceding %%writefile created /content/Noise_Aware_Tangent_Selector.py.
import Noise_Aware_Tangent_Selector as selection
results={}
for fam,file in bench_files.items():
    dest=RESULT_DIR/f'{fam}_selected'
    summary,report=selection.run(file,dest,alpha=ALPHA,tau=TRUE_ERROR_TAU,seed=SEED)
    report['benchmark_family']=fam
    (dest/'benchmark_protocol_context.json').write_text(json.dumps({'family':fam,'note':'Original selector report hardcodes circle; benchmark_family is authoritative for these executions','spectral_head':'Osada LHSD Soft_DoF exact D=8 trace','fixed_sigma':PRESET_FIXED_SIGMA,'c':LHSD_SOFT_C,'p':LHSD_SOFT_P},indent=2))
    results[fam]=(summary,report,dest)
    print(f'Completed synthetic calibration for {fam}; heldout simultaneous coverage:',report['observed_test_simultaneous_coverage'])

## 6. Official LHSD fixed-scale baseline and controlled comparisons
We do not claim that **fixed σ=0.128** is the published authors’ recommended optimal scale. It is an explicitly prespecified **application of the official LHSD Soft_DoF spectral head**. As scale-selection control, raw-proxy and raw-defect policies also share the same Q. Compare dimension errors separately from oracle-rank tangent errors.

In [ ]:
all_summary=[];all_point_rows=[]
for fam,file in bench_files.items():
    df=pd.read_csv(file);te=df[df.split=='test'].copy()
    selected=pd.read_csv(results[fam][2]/'test_selected_by_method.csv')
    names=['calibrated_upper_bound','noise_aware_predicted_error','raw_neighbor_proxy','raw_projector_defect']
    cal=selected[selected.method=='calibrated_upper_bound'].copy()
    accepted_ids=set(cal.loc[cal.accepted.astype(str).str.lower().isin(['true','1']),'index'].astype(int))
    selections={name:selected[selected.method==name].copy() for name in names}
    fixed=te[np.isclose(te.sigma,PRESET_FIXED_SIGMA)].copy()
    fixed['method']='LHSD_Soft_DoF_fixed_sigma'
    selections['LHSD_Soft_DoF_fixed_sigma']=fixed
    for name,z in selections.items():
        # Include the official LHSD dimension estimate associated with each choice.
        if 'dim_error' not in z.columns:
            z=z.merge(te[['index','sigma','dim_error']],on=['index','sigma'],how='left',validate='one_to_one')
        z['family']=fam
        if name=='calibrated_upper_bound':
            z['is_accepted']=z['accepted'].astype(str).str.lower().isin(['true','1'])
        else:
            z['is_accepted']=True
        z['on_calibrated_accepted_subset']=z['index'].isin(accepted_ids)
        all_point_rows.append(z.copy())
        for tag,zz in [('all_points',z),('common_calibrated_accepted_subset',z[z.on_calibrated_accepted_subset])]:
            acc=zz[zz.is_accepted]
            all_summary.append(dict(family=fam,method=name,subset=tag,n=len(zz),
                selected_subset_fraction=len(zz)/te['index'].nunique(),
                acceptance_coverage=len(acc)/te['index'].nunique(),
                mean_oracle_rank_tangent_error=zz.tangent_error.mean(),
                mean_oracle_rank_tangent_error_accepted=acc.tangent_error.mean(),
                mean_LHSD_dimension_MAE=zz.dim_error.mean(),
                mean_LHSD_dimension_MAE_accepted=acc.dim_error.mean(),
                fraction_accepted_tangent_error_above_tau=np.mean(acc.tangent_error>TRUE_ERROR_TAU) if len(acc) else np.nan))
summary=pd.DataFrame(all_summary)
points=pd.concat(all_point_rows,ignore_index=True)
summary.to_csv(RESULT_DIR/'BENCHMARK_SUMMARY.csv',index=False)
points.to_csv(RESULT_DIR/'ALL_POLICY_POINT_CHOICES.csv',index=False)
print('Benchmark all-points comparison:')
display(summary[summary.subset=='all_points'].sort_values(['family','mean_oracle_rank_tangent_error']))
print('Comparison on common calibrated accepted points (does not prove equal risk at matched independent acceptance rules):')
display(summary[summary.subset=='common_calibrated_accepted_subset'].sort_values(['family','mean_oracle_rank_tangent_error']))

## 7. Statistical contrasts, risk coverage and visualization
The paired bootstrap is over independent test points, **not over the three correlated sigma rows**. This is an exploratory within-family result; repeat with different network-training seeds to support a paper-level claim.

In [ ]:
import matplotlib.pyplot as plt
stat_rows=[]
fig,axs=plt.subplots(1,len(FAMILIES_TO_RUN),figsize=(6*len(FAMILIES_TO_RUN),4),squeeze=False)
for j,fam in enumerate(FAMILIES_TO_RUN):
    x=points[points.family==fam].pivot_table(index='index',columns='method',values='tangent_error')
    for base in ['LHSD_Soft_DoF_fixed_sigma','raw_neighbor_proxy']:
        delta=(x[base]-x['calibrated_upper_bound']).dropna().to_numpy()
        rng=np.random.default_rng(SEED+400+j)
        means=np.mean(delta[rng.integers(0,len(delta),size=(2000,len(delta)))],axis=1)
        interval=np.quantile(means,[.025,.975])
        stat_rows.append(dict(family=fam,baseline=base,n=len(delta),mean_baseline_minus_calibrated=float(delta.mean()),ci_low=float(interval[0]),ci_high=float(interval[1])))
    ax=axs[0,j]
    tab=summary[(summary.family==fam)&(summary.subset=='all_points')]
    ax.barh(tab.method,tab.mean_oracle_rank_tangent_error)
    ax.set_title(fam+' — oracle-rank tangent error')
    ax.set_xlabel('Lower is better')
    ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULT_DIR/'LHSD_vs_Calibrated_Results.png',dpi=160,bbox_inches='tight')
plt.show()
stat=pd.DataFrame(stat_rows)
stat.to_csv(RESULT_DIR/'PAIRED_CI95.csv',index=False)
display(stat)

## 8. Optional verification: run original official `Soft_DoF` implementation (not required for benchmark)
This clones the [official repository](https://github.com/geosada/LHSD) at its pinned June 18, 2026 commit and executes the **official function definitions from that checkout**, using a score-adapter for our own DSM network. It checks our exact-trace formula against an SLQ approximation on a single point. Since SLQ uses random probes, the values will not match exactly. **This cell is optional and can be skipped on CPU.**

In [ ]:
if VERIFY_OFFICIAL_SLQ:
    import subprocess, ast, types, torch
    PIN='fdd16ce2569ea74deed9a82f9b1d71d186d05492'
    repo=RESULT_DIR/'official_LHSD_checkout'
    if not (repo/'.git').exists():
        subprocess.run(['git','clone','--quiet','https://github.com/geosada/LHSD.git',str(repo)],check=True)
    subprocess.run(['git','-C',str(repo),'checkout','--quiet',PIN],check=True)
    assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==PIN
    sourcefile=repo/'lid/diffusions/local_hessian_spectrum.py'
    tree=ast.parse(sourcefile.read_text())
    needed={'hvp_logp_batch','_rademacher_probes','_general_slq_per_sample','Soft_DoF'}
    funcs=[n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in needed]
    assert {n.name for n in funcs}==needed
    ns={'torch':torch,'Sde':object}
    exec(compile(ast.Module(body=funcs,type_ignores=[]),str(sourcefile),'exec'),ns)
    class ScoreAdapter:
        def __init__(self,net):self.net=net
        def score(self,x,t):
            sig=t[:,None]
            return -self.net(x,sig)/sig
    fam=FAMILIES_TO_RUN[0]; net=get_model(fam)
    _,X,_,_=family_data(fam,'test'); sigma=SIGMAS[1]
    Q=q_batches(net,X[:1],sigma)
    exact=float(lhsd_soft_dof_from_Q(Q)[0])
    torch.manual_seed(43)
    with torch.enable_grad():
        approx=ns['Soft_DoF'](ScoreAdapter(net),torch.as_tensor(X[:1],dtype=torch.float32,device=DEVICE),
          t_scalar=sigma,kappa=LHSD_SOFT_C/(sigma*sigma),p=LHSD_SOFT_P,num_probe=12,lanczos_m=8)
    print('OFFICIAL pinned SHA:',PIN,'exact Soft_DoF=',exact,'official SLQ=',float(approx.item()),
          'absolute approximation difference=',abs(exact-float(approx.item())))
else:
    print('Official SLQ test skipped; primary benchmark uses mathematically equivalent exact spectral trace in D=8.')

## 9. Conclusions and research integrity

- **A positive result** requires reduced *held-out true tangent error* at **the same set of points**, reasonable LHSD dimension MAE, and intervals that do not systematically favor the comparator. Report coverage separately; abstention must not hide errors.
- **Conformal calibration** is only a marginal simultaneous error-bound claim for a held-out point from the calibrated synthetic family. It is not a conditional selective-risk guarantee, and cannot certify MNIST tangents.
- **Tangent extraction** here uses **known synthetic rank** for comparing scale choices, so it must not be presented as a fully unsupervised joint \(\hat d,\hat P_T\) recovery result.
- **Full official paper reproduction:** For publication, also run the authors’ experimental notebook against their native diffusion model/SLQ setup, with settings and resource requirements documented. Our experiment isolates the spectral head plus calibration on our score models; label it accurately.
- **Avoid tuning** `LHSD_SOFT_C`, `LHSD_SOFT_P`, sigma, risk tolerance, feature set, or model settings on the held-out test data.

Output folder: `MyDrive/LHSD_vs_Calibrated_Geometry_Benchmark`; copy its `BENCHMARK_SUMMARY.csv`, `PAIRED_CI95.csv`, `LHSD_vs_Calibrated_Results.png`, calibration files, and JSON reports into your proposal.